# Simple RNN · 07 The Simple RNN Model

Builds the Simple RNN architecture and counts its parameters. Nothing is trained here. **Saves:** nothing.

*Notebook 7 of 13 · Previous: 06 Class Weights · Next: 08 Training*

## Setup (the same in every notebook)

Each notebook in this project is self-contained. The cells below define the group settings and only the helper functions this notebook uses. They are identical in every notebook and nothing is imported from another file, so the notebook also runs on its own, for example in Colab.

In [ ]:
# ---- Imports, project folders and dataset location ----------------------------------------------
import sys, re, json, time, pathlib, textwrap
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from tensorflow import keras
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)

layers = keras.layers
IN_COLAB = "google.colab" in sys.modules

# RNN_ROOT is the folder that holds the numbered step folders (01_Setup, 02_Data_Audit, ...).
# If this notebook is opened on its own (for example in Colab), the current folder is used instead.
here = pathlib.Path.cwd().resolve()
RNN_ROOT = next((p for p in [here, *here.parents] if (p / "01_Setup").is_dir() or (p / "simple_rnn").is_dir()), here)
if (RNN_ROOT / "simple_rnn").is_dir():
    RNN_ROOT = RNN_ROOT / "simple_rnn"
RESULTS = RNN_ROOT / "results"              # figures, metrics, predictions and trained models are saved here
RESULTS.mkdir(exist_ok=True)

# The dataset is the only input file.
DATA_NAME = "deceptive-opinion.csv"
search_in = [RNN_ROOT, RNN_ROOT / "dataset", RNN_ROOT.parent / "dataset", here, here / "dataset",
             here.parent / "dataset", pathlib.Path("/content")]
DATA_PATH = next((d / DATA_NAME for d in search_in if (d / DATA_NAME).exists()), None)

if DATA_PATH is None and IN_COLAB:
    from google.colab import files
    print(f"Please choose {DATA_NAME} from your computer...")
    uploaded = files.upload()                       # saved in the current folder
    DATA_PATH = here / next(iter(uploaded))
if DATA_PATH is None:
    raise FileNotFoundError(f"Cannot find {DATA_NAME}. Put it in a 'dataset' folder next to the step folders "
                            "(or next to this notebook) and run this cell again.")

print("Dataset:", DATA_PATH)
print("Results folder:", RESULTS)

In [ ]:
# ---- Group settings: identical for Simple RNN, LSTM, GRU and 1D CNN ------------------------------
SEED = 42

# Labels: this dataset has no star rating, so the `polarity` column is the label.
LABEL_COLUMN = "polarity"
LABEL_MAP = {"negative": 0, "positive": 1}

# Text -> integer sequences
VOCAB_SIZE = 5000          # total ids, including <PAD>=0 and <OOV>=1
MAX_LEN = 300              # tokens per review after cleaning
PAD_TOKEN, OOV_TOKEN = "<PAD>", "<OOV>"
PADDING = "pre"            # zeros in front, so a recurrent layer ends on real words
TRUNCATING = "post"        # long reviews keep their first MAX_LEN tokens

# Training
EMBED_DIM = 64
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
MAX_EPOCHS = 30
PATIENCE = 5               # EarlyStopping patience, monitoring validation loss
MONITOR = "val_loss"
THRESHOLD = 0.5            # probability cut-off used for every model's test metrics

# Fixed 80/10/10 split (stratified on the label, seed 42, made after removing duplicates).
# Numbers are 0-based row positions in deceptive-opinion.csv. Every row not listed here is a TRAINING row.
# All four models must use exactly these two lists.
VAL_IDS = {
    3, 4, 6, 7, 14, 24, 25, 27, 33, 38, 46, 50, 60, 66, 72, 74, 77, 85, 89, 92, 106, 110, 112, 136,
    144, 172, 211, 215, 224, 230, 238, 242, 248, 252, 288, 302, 311, 319, 336, 345, 368, 376, 385,
    393, 402, 404, 434, 440, 442, 465, 473, 510, 512, 515, 517, 524, 533, 536, 542, 549, 553, 558,
    579, 581, 594, 603, 624, 630, 638, 644, 645, 661, 663, 665, 699, 713, 729, 754, 760, 788, 801,
    813, 820, 821, 827, 832, 840, 864, 873, 893, 897, 900, 904, 907, 908, 923, 930, 936, 962, 963,
    989, 991, 1002, 1008, 1017, 1043, 1045, 1054, 1073, 1098, 1112, 1119, 1134, 1143, 1170, 1189,
    1191, 1201, 1239, 1258, 1262, 1263, 1264, 1270, 1280, 1295, 1308, 1309, 1314, 1328, 1344, 1350,
    1359, 1360, 1364, 1367, 1368, 1369, 1370, 1377, 1378, 1381, 1418, 1423, 1450, 1467, 1479, 1480,
    1490, 1505, 1506, 1522, 1547, 1551, 1552, 1573, 1578, 1588, 1593, 1594
}
TEST_IDS = {
    0, 2, 5, 9, 39, 45, 47, 55, 108, 117, 120, 125, 138, 139, 145, 171, 191, 193, 213, 222, 227,
    228, 232, 256, 258, 260, 272, 273, 276, 281, 299, 314, 321, 325, 349, 362, 364, 391, 409, 424,
    436, 445, 467, 469, 484, 502, 503, 505, 507, 523, 525, 531, 545, 547, 572, 573, 605, 606, 612,
    622, 623, 633, 669, 671, 681, 687, 691, 698, 718, 725, 731, 741, 752, 753, 765, 770, 771, 779,
    784, 785, 804, 814, 834, 835, 847, 852, 859, 866, 882, 887, 889, 901, 932, 940, 958, 961, 968,
    972, 993, 1003, 1007, 1019, 1020, 1033, 1044, 1046, 1055, 1072, 1076, 1079, 1091, 1117, 1141,
    1147, 1149, 1151, 1176, 1182, 1183, 1193, 1195, 1196, 1205, 1210, 1217, 1222, 1231, 1259, 1265,
    1275, 1278, 1279, 1288, 1293, 1296, 1302, 1306, 1312, 1324, 1357, 1365, 1366, 1396, 1404, 1416,
    1446, 1451, 1465, 1478, 1485, 1487, 1492, 1503, 1504, 1516, 1524, 1540, 1562, 1582, 1597
}
keras.utils.set_random_seed(SEED)          # seeds Python, NumPy and TensorFlow
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| seed", SEED)
print("Devices:", [d.device_type for d in tf.config.list_physical_devices()])

## The Simple RNN Architecture

```
Input (300 token ids)
  → Embedding(5000, 64)        each word id → learned 64-number vector (320,000 params)
  → SimpleRNN(64)              standard Elman recurrent layer (8,256 params)
  → Dropout(0.5)               regularization preventing co-adaptation
  → Dense(32, ReLU)            non-linear projection (2,080 params)
  → Dropout(0.3)               regularization for dense layer
  → Dense(1, Sigmoid)          probability of positive review (33 params)
```

| Component | Choice | Justification |
|---|---|---|
| **Embedding(5000, 64)** | 5,000 vocab, dim 64 | Maps discrete token IDs to dense semantic vectors. Trained from scratch without pretrained weights to isolate architectural effects. |
| **SimpleRNN(64)** | 64 hidden units | Standard Elman RNN: $h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b_h)$. No gating mechanisms. |
| **Dropout(0.5)** | Rate 0.5 | Applied immediately after the recurrent layer to prevent memorization of the small 1,276-sample training set. |
| **Dense(32, ReLU)** | 32 units | Intermediate non-linear feature extractor before final classification. |
| **Dropout(0.3)** | Rate 0.3 | Mild regularizer before final output. |
| **Dense(1, Sigmoid)** | 1 unit, Sigmoid | Outputs $P(\text{positive}) \in [0, 1]$ paired with binary cross-entropy. |

In [ ]:
def build_simple_rnn():
    return keras.Sequential([
        layers.Input(shape=(MAX_LEN,)),
        layers.Embedding(VOCAB_SIZE, EMBED_DIM),
        layers.SimpleRNN(64),
        layers.Dropout(0.5),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid"),
    ], name="simple_rnn_sentiment")

model = build_simple_rnn()
model.summary()

In [ ]:
def n_params(weights):
    return int(sum(np.prod(w.shape) for w in weights))

breakdown = {layer.name: n_params(layer.trainable_weights) for layer in model.layers if layer.trainable_weights}
print("\nTrainable parameters per layer:", breakdown)
print(f"Total Parameters: {model.count_params():,}")
print(f"Embedding share of all parameters: {100 * breakdown['embedding'] / model.count_params():.2f}%")
print(f"Simple RNN layer parameters: {breakdown['simple_rnn']:,} (W_xh: 64*64=4096, W_hh: 64*64=4096, bias: 64)")

---
*Notebook 7 of 13 · Previous: 06 Class Weights · Next: 08 Training*